# Describing Study Sessions

*Combine center, spread, quartiles, and paired association in a reproducible descriptive report.*


## Prepared Study-Session Observations

Eleven prepared records each describe one student's study session and quiz
score. Ten sessions have recorded durations; one duration is missing. The
120-minute session is unusually long but remains a valid observed duration.
Scores are on a 100-point scale. These illustrative records are not a random
sample supporting claims about all students.

Summaries exclude missing values separately for each variable. Sample standard
deviation uses `ddof=1`; quartiles use linear interpolation. Correlation uses
only rows with both fields observed. IQR flags are retained for inspection,
and a comparison with unflagged durations is labeled as a sensitivity analysis.


## Imports, Records, and Saved Result Paths

The notebook includes every input. Output paths are relative to the chapter
directory locally and to `/home/pyodide` in the course site's browser filesystem.
Each run replaces the same report files.


In [1]:
import os
import numpy as np
import pandas as pd

prepared_rows = {
    "student_id": list(range(1, 12)),
    "study_minutes": [20, 25, 30, 30, 35, 40, 45, 50, 55, 120, np.nan],
    "quiz_score": [50, 55, 60, 60, 65, 70, 75, 80, 85, 90, 65],
}
output_directory = os.path.join("outputs", "08_study_sessions")
summary_output_path = os.path.join(output_directory, "descriptive_summary.csv")
records_output_path = os.path.join(output_directory, "session_flags.csv")
os.makedirs(output_directory, exist_ok=True)
print(pd.DataFrame(prepared_rows).to_string(index=False))


 student_id  study_minutes  quiz_score
          1           20.0          50
          2           25.0          55
          3           30.0          60
          4           30.0          60
          5           35.0          65
          6           40.0          70
          7           45.0          75
          8           50.0          80
          9           55.0          85
         10          120.0          90
         11            NaN          65


Student 11 has a quiz score but no recorded study duration. Student 10
has the longest recorded session, at 120 minutes.


## Summarize Each Variable with Its Own Count

`summarize_variable()` explicitly selects observed values and returns center,
sample spread, and quartiles. `count` and `missing` expose how many records
contributed. The original units remain in the variable names.


In [2]:
def summarize_variable(series):
    observed = series.dropna().to_numpy(dtype=float)
    q1, median, q3 = np.quantile(observed, [0.25, 0.5, 0.75], method="linear")
    return {
        "variable": series.name,
        "count": len(observed),
        "missing": int(series.isna().sum()),
        "mean": observed.mean(),
        "median": median,
        "sample_std": observed.std(ddof=1),
        "minimum": observed.min(),
        "q1": q1,
        "q3": q3,
        "maximum": observed.max(),
    }


summary_df = pd.DataFrame(
    [
        summarize_variable(pd.Series(prepared_rows[column], name=column))
        for column in ["study_minutes", "quiz_score"]
    ]
)
print(summary_df.round(3).to_string(index=False))


     variable  count  missing   mean  median  sample_std  minimum   q1    q3  maximum
study_minutes     10        1 45.000    37.5      28.577     20.0 30.0 48.75    120.0
   quiz_score     11        0 68.636    65.0      12.667     50.0 60.0 77.50     90.0


The duration summary uses 10 observations and reports one missing value. Its mean is 45 minutes, median 37.5 minutes, and sample standard deviation about 28.577 minutes. The quiz summary uses all 11 scores.


## Preserve IQR Flags and Paired Observations

Compute duration fences from observed durations, then attach the flags to
the original rows. A missing duration receives no outlier flag and remains
explicitly missing. Drop missing rows only in the paired view used to calculate
correlation.


In [3]:
def inspect_sessions(records):
    result = records.copy()
    q1, q3 = result["study_minutes"].quantile([0.25, 0.75], interpolation="linear")
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    result["iqr_flag"] = (result["study_minutes"] < lower) | (
        result["study_minutes"] > upper
    )
    paired = result[["study_minutes", "quiz_score"]].dropna()
    correlation = paired["study_minutes"].corr(paired["quiz_score"])
    return result, paired, correlation, (lower, upper)


flagged_df, paired_df, paired_correlation, fences = inspect_sessions(
    pd.DataFrame(prepared_rows)
)
print("Duration fences:", fences)
print("Flagged records:")
print(flagged_df.loc[flagged_df["iqr_flag"]].to_string(index=False))
print("Complete pairs:", len(paired_df))
print(f"Study-duration and quiz correlation: {paired_correlation:.4f}")
print(
    f"Mean duration with all observed sessions: {flagged_df['study_minutes'].mean():.3f}"
)
print(
    f"Mean duration excluding flagged sessions: {flagged_df.loc[~flagged_df['iqr_flag'], 'study_minutes'].mean():.3f}"
)


Duration fences: (1.875, 76.875)
Flagged records:
 student_id  study_minutes  quiz_score  iqr_flag
         10          120.0          90      True
Complete pairs: 10
Study-duration and quiz correlation: 0.8337
Mean duration with all observed sessions: 45.000
Mean duration excluding flagged sessions: 36.667


Student 10 is the only IQR flag. The 10 complete pairs have correlation
about 0.8337. Excluding the long session changes mean duration from 45 to
36.667 minutes; that is a different subset, not a corrected truth.


## Save and Verify the Two Tables

Write a summary and a row-level flag table. Floating-point CSV conversions can
differ in their last digits, so the summary comparison permits small numerical
rounding differences. `float_precision="round_trip"` preserves the saved
floating values when reloading the summary.


In [4]:
def save_and_verify(summary, records):
    summary.to_csv(summary_output_path, index=False, encoding="utf-8")
    records.to_csv(records_output_path, index=False, encoding="utf-8")
    reopened_summary = pd.read_csv(
        summary_output_path, encoding="utf-8", float_precision="round_trip"
    )
    reopened_records = pd.read_csv(records_output_path, encoding="utf-8")
    summary_matches = reopened_summary["variable"].equals(
        summary["variable"]
    ) and np.allclose(
        reopened_summary.drop(columns="variable"),
        summary.drop(columns="variable"),
        equal_nan=True,
    )
    print("Saved summary matches:", summary_matches)
    print("Saved records match:", reopened_records.equals(records))
    print("Saved files:", summary_output_path, records_output_path)
    return reopened_summary, reopened_records


## Run the Complete Descriptive Report

`main()` rebuilds the records, computes the summaries, attaches inspection
flags, and verifies the saved results. It does not depend on any guided notebook.


In [5]:
def main():
    records = pd.DataFrame(prepared_rows)
    summary = pd.DataFrame(
        [
            summarize_variable(records[column])
            for column in ["study_minutes", "quiz_score"]
        ]
    )
    flagged, paired, correlation, fences = inspect_sessions(records)
    print(summary.round(3).to_string(index=False))
    print("Observed duration count:", records["study_minutes"].count())
    print(
        "Flagged student IDs:", flagged.loc[flagged["iqr_flag"], "student_id"].tolist()
    )
    print("Paired count:", len(paired))
    print(f"Paired correlation: {correlation:.4f}")
    save_and_verify(summary, flagged)
    return summary, flagged


final_summary_df, final_flagged_df = main()


     variable  count  missing   mean  median  sample_std  minimum   q1    q3  maximum
study_minutes     10        1 45.000    37.5      28.577     20.0 30.0 48.75    120.0
   quiz_score     11        0 68.636    65.0      12.667     50.0 60.0 77.50     90.0
Observed duration count: 10
Flagged student IDs: [10]
Paired count: 10
Paired correlation: 0.8337
Saved summary matches: True
Saved records match: True
Saved files: outputs/08_study_sessions/descriptive_summary.csv outputs/08_study_sessions/session_flags.csv


## Result Analysis

The 11 records contain **10 observed durations** and **11 observed quiz scores**.
Observed duration has mean **45 minutes**, median **37.5 minutes**, and sample
standard deviation **28.577 minutes**. Linear Q1 and Q3 are **30** and **48.75**,
so the IQR is **18.75 minutes** and the fences are **1.875** and **76.875**.

Student **10**, with a 120-minute session, is the only flagged record. Removing
that observation from a sensitivity calculation lowers the duration mean to
**36.667 minutes**; the saved record table retains it. The **10 complete pairs**
have Pearson correlation approximately **0.8337**. This describes positive linear
association in the prepared observations, without establishing a causal effect.

Both saved-table comparisons are `True`. The report preserves denominators,
missingness, and flags, and repeated runs replace the same output files.
